In [1]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__)
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F
from urllib.parse import urlparse
from datetime import date
from utils import find_new_paths

3.4.1


In [2]:
spark = (SparkSession.builder
    .appName("preprocess_batch")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/nifi/crypto-prices")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.hadoop.hive.exec.dynamic.partition", "true")
    .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
    .getOrCreate())

spark.sql("USE DATABASE CryptoPredictions")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/16 20:10:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/16 20:10:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/16 20:10:50 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

# Quality Test

In [8]:
def test_nulls(df):
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()
    
df = spark.read.parquet("hdfs://namenode:8020/nifi/crypto-prices/crypto-quotes-20251106-214622.parquet")
test_nulls(df)

[Stage 1:>                                                          (0 + 1) / 1]

+-------------+----+---+----+--------------+---------+------+
|current_price|high|low|open|previous_close|timestamp|symbol|
+-------------+----+---+----+--------------+---------+------+
|            0|   0|  0|   0|             0|        0|     0|
+-------------+----+---+----+--------------+---------+------+



In [9]:
from pyspark.sql.types import *
expected_schema = StructType([
    StructField("current_price", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("open", DoubleType(), True),
    StructField("previous_close", DoubleType(), True),
    StructField("timestamp", IntegerType(), True),
    StructField("symbol", StringType(), True)
])
def test_datatypes(df, expected_schema):
    return df.schema == expected_schema

In [10]:
test_datatypes(df, expected_schema)

True

# Preprocessing całego folderu i zapis do Hive

In [3]:
BASE_PATH = "hdfs://namenode:8020/nifi/crypto-prices/"
META_PATH = "hdfs://namenode:8020/nifi/metadata/crypto_prices_last_path.txt"

paths_to_read = find_new_paths(spark, BASE_PATH, META_PATH)

No new data.


In [16]:
def rename_and_transform(df):
    # Wyciągnięcie symbolu
    df = df.withColumn(
        "Symbol",
        F.split("symbol", ":")[1].substr(1, 3)
    )

    # Timestamp → Datetime
    df = df.withColumn(
        "Datetime",
        F.from_unixtime(F.col("timestamp")).cast("timestamp")
    )
    
    # Mapowanie: stara_nazwa → nowa_nazwa
    rename_map = {
        "current_price": "CurrentPrice",
        "open": "OpeningPrice",
        "low": "LowestDayPrice",
        "high": "HighestDayPrice",
        "previous_close": "PreviousClosingPrice"
    }

    # Zmiana nazw
    df = df.withColumnsRenamed(rename_map)
    
    df = df.withColumn("PartitionDate", F.to_date("Datetime"))
    df= df.filter(df.PartitionDate.isNotNull())
    
    return df.select(
        "Symbol",
        "CurrentPrice",
        "OpeningPrice",
        "LowestDayPrice",
        "HighestDayPrice",
        "PreviousClosingPrice",
        "Datetime",
        "PartitionDate"
    )

df = spark.read.parquet(*paths_to_read)
df = rename_and_transform(df)

# print(df.count())
df.describe().show()
df.sort(df.Datetime.desc()).show(5)

+-------+------+------------------+------------------+------------------+------------------+--------------------+
|summary|Symbol|      CurrentPrice|      OpeningPrice|    LowestDayPrice|   HighestDayPrice|PreviousClosingPrice|
+-------+------+------------------+------------------+------------------+------------------+--------------------+
|  count|316757|            316757|            316757|            316757|            316757|              316757|
|   mean|  null|31245.224916829004|31284.765572757668|30655.853417254988|31856.249760889918|  31284.765291153817|
| stddev|  null| 41943.23428708404| 41998.99820916887|41174.981193716616| 42742.79699011492|   41998.99800460141|
|    min|   BTC|             123.2|            123.23|            123.11|            128.54|              123.23|
|    max|   SOL|          94584.89|          94581.99|           91801.0|          94588.99|             94582.0|
+-------+------+------------------+------------------+------------------+---------------

[Stage 26:==============================================>         (15 + 1) / 18]

+------+------------+------------+--------------+---------------+--------------------+-------------------+-------------+
|Symbol|CurrentPrice|OpeningPrice|LowestDayPrice|HighestDayPrice|PreviousClosingPrice|           Datetime|PartitionDate|
+------+------------+------------+--------------+---------------+--------------------+-------------------+-------------+
|   SOL|      127.81|      129.35|        123.63|         135.43|              129.34|2025-12-15 23:59:51|   2025-12-15|
|   ETH|     2964.85|     3062.47|       2894.57|         3177.5|             3062.48|2025-12-15 23:59:51|   2025-12-15|
|   BTC|    86432.07|     88158.0|      85146.64|       90052.64|            88158.01|2025-12-15 23:59:51|   2025-12-15|
|   BTC|    86426.31|    88151.93|      85146.64|       90052.64|            88151.93|2025-12-15 23:59:31|   2025-12-15|
|   ETH|     2964.85|      3060.0|       2894.57|         3177.5|             3060.01|2025-12-15 23:59:31|   2025-12-15|
+------+------------+-----------

In [17]:
(df.write
  .mode("append")
  .format("hive")
  .partitionBy("PartitionDate")
  .saveAsTable("CryptocurrencySnapshot"))

latest_dir = max(new_dirs)

spark.createDataFrame(
    [(latest_dir,)],
    ["last_processed_path"]
).write.mode("overwrite").text(META_PATH)

print(f"Created last path file: {latest_dir}")

25/12/16 18:15:44 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
                                                                                

Created last path file.


In [18]:
spark.sql("""SELECT * FROM CryptocurrencySnapshot
            ORDER BY PartitionDate DESC
            LIMIT 10""").show()

[Stage 30:=================================================>      (16 + 1) / 18]

+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|Symbol|           Datetime|CurrentPrice|OpeningPrice|LowestDayPrice|HighestDayPrice|PreviousClosingPrice|PartitionDate|
+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|   BTC|2025-12-15 02:25:07|    89737.47|    90370.51|      87577.36|        90384.0|            90370.51|   2025-12-15|
|   ETH|2025-12-15 10:45:04|     3157.88|      3106.2|       3024.44|         3177.5|              3106.2|   2025-12-15|
|   BTC|2025-12-15 20:45:46|    85914.16|    88664.22|      85146.64|       90052.64|            88664.21|   2025-12-15|
|   ETH|2025-12-15 10:46:01|     3156.59|     3106.66|       3024.44|         3177.5|             3106.65|   2025-12-15|
|   SOL|2025-12-15 18:15:41|      125.22|      130.37|        124.51|         135.43|              130.37|   2025-12-15|
|   BTC|2025-12-15 10:45:04|    

In [30]:
spark.stop()